# UC08 — Extracción Documental IA en Siniestros

Pipeline de extracción de datos estructurados desde documentos de siniestros con Cortex AI.
**Valor de negocio**: Reducir el tiempo de tramitación documental un 70%.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS EXTRACCION_DOCS_DB;
USE SCHEMA EXTRACCION_DOCS_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Documentos Sintéticos (200)

In [ ]:
CREATE OR REPLACE TABLE documentos_siniestros AS
SELECT 'DOC' || LPAD(SEQ4(),5,'0') AS doc_id, 'SIN' || LPAD(MOD(SEQ4(),50)+1,5,'0') AS siniestro_id,
    CASE MOD(SEQ4(),4) WHEN 0 THEN 'Atestado_Policial' WHEN 1 THEN 'Factura_Medica' WHEN 2 THEN 'Presupuesto_Reparacion' ELSE 'Declaracion_Amistosa' END AS tipo_doc,
    CASE MOD(SEQ4(),4)
        WHEN 0 THEN 'Atestado policial del accidente ocurrido el ' || DATEADD(day,-UNIFORM(1,365,RANDOM()),CURRENT_DATE())::VARCHAR || ' en la calle ' || CASE MOD(SEQ4(),5) WHEN 0 THEN 'Gran Vía, Madrid' WHEN 1 THEN 'Diagonal, Barcelona' WHEN 2 THEN 'Colón, Valencia' WHEN 3 THEN 'Constitución, Sevilla' ELSE 'Alameda, Málaga' END || '. Culpabilidad: ' || CASE WHEN UNIFORM(0,1,RANDOM())<0.5 THEN 'del segundo conductor.' ELSE 'compartida.' END
        WHEN 1 THEN 'Factura médica por asistencia de urgencias. Paciente atendido por ' || CASE MOD(SEQ4(),3) WHEN 0 THEN 'cervicalgia postraumática' WHEN 1 THEN 'contusión en extremidades' ELSE 'latigazo cervical' END || '. Importe: ' || UNIFORM(150,3000,RANDOM()) || '€.'
        WHEN 2 THEN 'Presupuesto de reparación del vehículo matrícula ' || UNIFORM(1000,9999,RANDOM()) || 'GHI. Daños: ' || CASE MOD(SEQ4(),3) WHEN 0 THEN 'paragolpes trasero' WHEN 1 THEN 'puerta lateral' ELSE 'capó y parabrisas' END || '. Importe total: ' || UNIFORM(500,8000,RANDOM()) || '€.'
        ELSE 'Declaración amistosa de accidente. Lugar: ' || CASE MOD(SEQ4(),4) WHEN 0 THEN 'rotonda' WHEN 1 THEN 'cruce' WHEN 2 THEN 'aparcamiento' ELSE 'autopista' END || '. ' || CASE MOD(SEQ4(),3) WHEN 0 THEN 'Colisión por alcance en semáforo.' WHEN 1 THEN 'Roce lateral al cambiar de carril.' ELSE 'Impacto al salir de aparcamiento.' END
    END AS texto_contenido
FROM TABLE(GENERATOR(ROWCOUNT=>200));

---

## Paso 3: Extraer Entidades con Cortex AI

In [ ]:
CREATE OR REPLACE TABLE datos_extraidos AS
SELECT doc_id, siniestro_id, tipo_doc,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Extrae los siguientes campos del texto y devuelve SOLO un JSON válido: {"fecha":"","lugar":"","matriculas":[],"lesiones":"si/no","importe_estimado":0,"culpabilidad":"asegurado/tercero/compartida/desconocida"}. Texto: ' || texto_contenido
    ) AS extraccion_raw
FROM documentos_siniestros;

SELECT doc_id, tipo_doc, LEFT(extraccion_raw,200) AS preview FROM datos_extraidos LIMIT 10;

---

## Paso 4: Clasificar Documentos Automáticamente

In [ ]:
CREATE OR REPLACE TABLE clasificacion_docs AS
SELECT doc_id, tipo_doc AS tipo_real,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Clasifica este texto en exactamente UNA categoría, responde solo con la categoría: Atestado_Policial, Factura_Medica, Presupuesto_Reparacion, Declaracion_Amistosa. Texto: ' || texto_contenido
    ) AS tipo_predicho
FROM documentos_siniestros;

SELECT tipo_real, tipo_predicho, COUNT(*) AS docs FROM clasificacion_docs GROUP BY tipo_real, tipo_predicho ORDER BY tipo_real;

---

## Paso 5: Vista de Siniestros Enriquecidos

In [ ]:
CREATE OR REPLACE VIEW siniestros_enriquecidos AS
SELECT siniestro_id, COUNT(*) AS total_docs,
    SUM(CASE WHEN tipo_doc='Atestado_Policial' THEN 1 ELSE 0 END) AS tiene_atestado,
    SUM(CASE WHEN tipo_doc='Factura_Medica' THEN 1 ELSE 0 END) AS tiene_factura_medica,
    SUM(CASE WHEN tipo_doc='Presupuesto_Reparacion' THEN 1 ELSE 0 END) AS tiene_presupuesto,
    CASE WHEN COUNT(*)>=2 THEN 'Completo' ELSE 'Pendiente' END AS estado_documental
FROM documentos_siniestros GROUP BY siniestro_id;

SELECT * FROM siniestros_enriquecidos ORDER BY total_docs DESC LIMIT 15;

---

## Paso 6: Dashboard Interactivo

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('Extracción Documental IA en Siniestros')

docs = session.table('documentos_siniestros').to_pandas()
sinr = session.sql('SELECT * FROM siniestros_enriquecidos').to_pandas()

c1,c2,c3 = st.columns(3)
with c1: st.metric('Documentos Procesados', len(docs))
with c2: st.metric('Siniestros', len(sinr))
with c3: st.metric('Completos', len(sinr[sinr['ESTADO_DOCUMENTAL']=='Completo']))

st.markdown('---')
st.subheader('Documentos por Tipo')
tc = docs['TIPO_DOC'].value_counts()
st.bar_chart(pd.DataFrame({'Docs': tc.values}, index=tc.index))

st.markdown('---')
st.subheader('Estado Documental por Siniestro')
st.dataframe(sinr, use_container_width=True, height=400)
st.caption('Desarrollado con Snowflake Cortex AI + Streamlit')

---

## Paso 7: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS EXTRACCION_DOCS_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;